# xG & PSxG — Full Model Zoo (StatsBomb Open Data)

This notebook has two independent parts:

**Part A — Get the data (optional).** Downloads free StatsBomb shot events and
builds engineered features. Skip this if you already have a shots CSV (e.g.
one you downloaded before and are re-uploading to this Colab session).

**Part B — Load the dataset.** Looks for the CSV in this Colab session's
files; if it isn't there, prompts you to upload it via the Files picker.
Works whether it's the full-feature file Part A produces, or a lighter CSV
you already have (like `xg_psxg_shots.csv`) — it adapts to whatever columns
are actually present.

**Part C — Model Zoo.** Trains and compares, on the *same* features:
Logistic Regression, Random Forest, Gradient Boosting, XGBoost, LightGBM,
SVM, K-Nearest Neighbors, Naive Bayes, a small Neural Net (MLP), a Stacked
Ensemble, and a Calibrated version of the best model.

**Part D — Save everything.** Metrics table, predictions, trained models,
and every chart get written to `/content/results/` and zipped for download.

**Part E — Bonus.** Shot clustering (unsupervised) and a note on where
computer vision actually fits into xG work (spoiler: not on this CSV
directly — it needs raw video or freeze-frame data, explained below).

## 1. Install dependencies

In [ ]:
!pip install -q statsbombpy pandas scikit-learn matplotlib seaborn xgboost lightgbm joblib


In [ ]:
import warnings
from statsbombpy.api_client import NoAuthWarning
warnings.filterwarnings('ignore', category=NoAuthWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import os
import ast
import json
import shutil
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')

RESULTS_DIR = '/content/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to {RESULTS_DIR}")

## 2. Part A — Get the data (optional)

Set `DOWNLOAD_FRESH_DATA = True` to pull data straight from StatsBomb.
Set it to `False` (default) if you already have a CSV and just want to
upload it in Part B below — this skips straight past Part A.

In [ ]:
DOWNLOAD_FRESH_DATA = False  # True = pull fresh data from StatsBomb. False = skip to Part B and upload an existing CSV.

COMPETITIONS = [
    (43, 3),    # FIFA World Cup 2018
    (43, 106),  # FIFA World Cup 2022
    (72, 30),   # Women's World Cup 2019
]

FULL_FEATURES_PATH = 'shots_full_features.csv'

if DOWNLOAD_FRESH_DATA:
    from statsbombpy import sb

    all_matches = []
    for comp_id, season_id in COMPETITIONS:
        try:
            m = sb.matches(competition_id=comp_id, season_id=season_id)
            m['competition_id'] = comp_id
            m['season_id'] = season_id
            all_matches.append(m)
            print(f"comp {comp_id} / season {season_id}: {len(m)} matches")
        except Exception as e:
            print(f"Could not load comp {comp_id}/season {season_id}: {e}")
    matches = pd.concat(all_matches, ignore_index=True)
    print(f"Total matches: {len(matches)}")
else:
    print("Skipping download. Go to Part B and upload your existing CSV.")

In [ ]:
if DOWNLOAD_FRESH_DATA:
    CACHE_FILE = 'raw_shots.csv'
    if os.path.exists(CACHE_FILE):
        shots_df = pd.read_csv(CACHE_FILE)
        print(f"Loaded cached {CACHE_FILE} with {len(shots_df)} shots")
    else:
        all_shots, failed = [], []
        for i, row in matches.iterrows():
            match_id = row['match_id']
            try:
                events = sb.events(match_id=match_id)
                s = events[events['type'] == 'Shot'].copy()
                s['match_id'] = match_id
                s['competition_id'] = row['competition_id']
                all_shots.append(s)
            except Exception:
                failed.append(match_id)
            if (i + 1) % 20 == 0:
                print(f"Processed {i+1}/{len(matches)} matches")
        shots_df = pd.concat(all_shots, ignore_index=True)
        shots_df.to_csv(CACHE_FILE, index=False)
        print(f"Downloaded {len(shots_df)} shots ({len(failed)} matches skipped)")

### Feature engineering (Part A) — builds the *full* feature set, including placement features needed for a proper PSxG model zoo

In [ ]:
if DOWNLOAD_FRESH_DATA:
    def parse_loc(val):
        if isinstance(val, str):
            try:
                return ast.literal_eval(val)
            except (ValueError, SyntaxError):
                return None
        return val

    df_raw = shots_df.copy()
    df_raw['location'] = df_raw['location'].apply(parse_loc)
    df_raw = df_raw[df_raw['location'].notna()].copy()
    df_raw['x'] = df_raw['location'].apply(lambda l: l[0])
    df_raw['y'] = df_raw['location'].apply(lambda l: l[1])

    GOAL_X, GOAL_Y, GOAL_WIDTH = 120, 40, 7.32
    df_raw['distance'] = np.sqrt((GOAL_X - df_raw['x'])**2 + (GOAL_Y - df_raw['y'])**2)
    df_raw['angle'] = np.abs(np.arctan2(
        GOAL_WIDTH * (GOAL_X - df_raw['x']),
        (GOAL_X - df_raw['x'])**2 + (df_raw['y'] - GOAL_Y)**2 - (GOAL_WIDTH / 2)**2
    ))
    df_raw['angle_deg'] = np.degrees(df_raw['angle'])

    df_raw['is_header'] = (df_raw['shot_body_part'] == 'Head').astype(int)
    df_raw['is_left_foot'] = (df_raw['shot_body_part'] == 'Left Foot').astype(int)
    df_raw['is_right_foot'] = (df_raw['shot_body_part'] == 'Right Foot').astype(int)
    df_raw['is_open_play'] = (df_raw['shot_type'] == 'Open Play').astype(int)
    df_raw['is_set_piece'] = (df_raw['shot_type'].isin(['Free Kick', 'Corner'])).astype(int)
    df_raw['is_penalty'] = (df_raw['shot_type'] == 'Penalty').astype(int)
    df_raw['under_pressure'] = df_raw.get('under_pressure', pd.Series(False, index=df_raw.index)).fillna(False).astype(int)
    df_raw['goal'] = (df_raw['shot_outcome'] == 'Goal').astype(int)

    def parse_end_loc(val):
        parsed = parse_loc(val)
        if parsed is None:
            return None, None, None
        if len(parsed) == 2:
            return parsed[0], parsed[1], 0.0
        return parsed[0], parsed[1], parsed[2]

    end_cols = df_raw['shot_end_location'].apply(parse_end_loc)
    df_raw['end_x'] = end_cols.apply(lambda t: t[0])
    df_raw['end_y'] = end_cols.apply(lambda t: t[1])
    df_raw['end_z'] = end_cols.apply(lambda t: t[2])

    GOAL_POST_TOP_LEFT_Y, GOAL_POST_TOP_RIGHT_Y = 36, 44
    df_raw['dist_to_nearest_post'] = np.minimum(
        np.abs(df_raw['end_y'] - GOAL_POST_TOP_LEFT_Y),
        np.abs(df_raw['end_y'] - GOAL_POST_TOP_RIGHT_Y)
    )
    df_raw['placement_height'] = df_raw['end_z'].fillna(0)
    df_raw['on_target'] = ((df_raw['end_y'].between(36, 44)) & (df_raw['placement_height'] < 2.67)).astype(int)

    keep_cols = ['player','team','match_id','competition_id','minute','x','y','distance','angle_deg',
                 'shot_body_part','shot_type','shot_outcome','goal','shot_statsbomb_xg',
                 'is_header','is_left_foot','is_right_foot','is_open_play','is_set_piece','is_penalty',
                 'under_pressure','end_x','end_y','end_z','dist_to_nearest_post','placement_height','on_target']
    keep_cols = [c for c in keep_cols if c in df_raw.columns]
    export_full = df_raw[keep_cols].rename(columns={'shot_statsbomb_xg': 'statsbomb_xg'})
    export_full.to_csv(FULL_FEATURES_PATH, index=False)
    print(f"Saved {len(export_full)} rows with full features to {FULL_FEATURES_PATH}")

## 3. Part B — Load the dataset

This is where the notebook picks up if you skipped Part A. It looks for a
CSV already sitting in this Colab session (from Part A, or from a previous
upload earlier in this session). If it doesn't find one, it opens the
**Colab file picker** so you can upload a CSV from your computer — e.g. the
`xg_psxg_shots.csv` you already downloaded before.

Either schema works: the full-feature file Part A produces, or the lighter
`xg_psxg_shots.csv` (columns: player, team, match_id, minute, x, y,
distance, angle_deg, shot_body_part, shot_type, shot_outcome, goal,
statsbomb_xg, my_xg, my_psxg). The next cell figures out what you gave it.

In [ ]:
CANDIDATE_PATHS = ['shots_full_features.csv', 'xg_psxg_shots.csv']
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if DATA_PATH is None:
    print("No dataset found in this Colab session.")
    print("Upload your CSV now (e.g. xg_psxg_shots.csv or shots_full_features.csv):")
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        DATA_PATH = input('Enter the path to your CSV: ').strip()
else:
    print(f"Found existing dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows, {df.shape[1]} columns from {DATA_PATH}")
df.head()

In [ ]:
# Make sure the flag columns exist regardless of which schema was uploaded
if 'is_header' not in df.columns:
    df['is_header'] = (df['shot_body_part'] == 'Head').astype(int)
    df['is_left_foot'] = (df['shot_body_part'] == 'Left Foot').astype(int)
    df['is_right_foot'] = (df['shot_body_part'] == 'Right Foot').astype(int)
    df['is_open_play'] = (df['shot_type'] == 'Open Play').astype(int)
    df['is_set_piece'] = (df['shot_type'].isin(['Free Kick', 'Corner'])).astype(int)
    df['is_penalty'] = (df['shot_type'] == 'Penalty').astype(int)
if 'under_pressure' not in df.columns:
    df['under_pressure'] = 0

PLACEMENT_COLS = ['end_x', 'end_y', 'dist_to_nearest_post', 'placement_height', 'on_target']
HAS_PLACEMENT = all(c in df.columns for c in PLACEMENT_COLS)

print(f"Total shots: {len(df)} | Goals: {int(df['goal'].sum())} ({df['goal'].mean():.1%})")
print(f"Placement features available for a full PSxG model zoo: {HAS_PLACEMENT}")
if not HAS_PLACEMENT:
    print("This CSV doesn't include end_x/end_y/placement columns, so the PSxG model-zoo")
    print("section below will be skipped. To enable it, set DOWNLOAD_FRESH_DATA=True in")
    print("Part A and rerun — the exported CSV there includes these columns.")

## 4. Part C — Model Zoo

Same idea for both tasks below: split the data, fit every model on the
identical training set, score every model on the identical held-out test
set, and compare fairly. Metrics used:

- **ROC AUC** — how well the model ranks goals above non-goals (higher is better, 0.5 = random)
- **Brier score** — mean squared error of the probability itself (lower is better)
- **Log loss** — penalizes confidently wrong predictions more (lower is better)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, brier_score_loss, log_loss, roc_curve,
                              confusion_matrix, ConfusionMatrixDisplay)

PENALTY_XG = 0.76  # standard reference value; penalties are excluded from training below

def build_model_zoo():
    """Fresh, unfitted copies of every model — called once per task so tasks don't share fitted state."""
    models = {
        'Logistic Regression': Pipeline([('scale', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
        'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42),
        'SVM (RBF)': Pipeline([('scale', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
        'K-Nearest Neighbors': Pipeline([('scale', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=25))]),
        'Naive Bayes': GaussianNB(),
        'Neural Net (MLP)': Pipeline([('scale', StandardScaler()),
                                      ('clf', MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42))]),
    }
    try:
        from xgboost import XGBClassifier
        models['XGBoost'] = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                           eval_metric='logloss', random_state=42, verbosity=0)
    except ImportError:
        print("xgboost not installed — skipping")
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM'] = LGBMClassifier(n_estimators=300, max_depth=4, random_state=42, verbosity=-1)
    except ImportError:
        print("lightgbm not installed — skipping")
    return models

def run_model_zoo(X, y, task_name):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    models = build_model_zoo()
    results, fitted, roc_data = [], {}, {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        results.append({'task': task_name, 'model': name,
                        'roc_auc': roc_auc_score(y_test, proba),
                        'brier': brier_score_loss(y_test, proba),
                        'log_loss': log_loss(y_test, proba),
                        'n_train': len(X_train), 'n_test': len(X_test)})
        fitted[name] = model
        roc_data[name] = roc_curve(y_test, proba)[:2]

    # Stacked ensemble
    stack_base = [(n, models[n]) for n in ['Logistic Regression', 'Random Forest'] if n in models]
    if 'XGBoost' in models:
        stack_base.append(('XGBoost', build_model_zoo()['XGBoost']))
    stack = StackingClassifier(estimators=stack_base, final_estimator=LogisticRegression(max_iter=1000), cv=3)
    stack.fit(X_train, y_train)
    proba = stack.predict_proba(X_test)[:, 1]
    results.append({'task': task_name, 'model': 'Stacked Ensemble',
                     'roc_auc': roc_auc_score(y_test, proba), 'brier': brier_score_loss(y_test, proba),
                     'log_loss': log_loss(y_test, proba), 'n_train': len(X_train), 'n_test': len(X_test)})
    fitted['Stacked Ensemble'] = stack
    roc_data['Stacked Ensemble'] = roc_curve(y_test, proba)[:2]

    # Calibrated version of whichever model scored best on ROC AUC
    results_so_far = pd.DataFrame(results)
    best_name = results_so_far.loc[results_so_far['roc_auc'].idxmax(), 'model']
    base_for_cal = build_model_zoo().get(best_name, LogisticRegression(max_iter=1000))
    calibrated = CalibratedClassifierCV(base_for_cal, method='isotonic', cv=3)
    calibrated.fit(X_train, y_train)
    proba = calibrated.predict_proba(X_test)[:, 1]
    cal_name = f'{best_name} (Calibrated)'
    results.append({'task': task_name, 'model': cal_name,
                     'roc_auc': roc_auc_score(y_test, proba), 'brier': brier_score_loss(y_test, proba),
                     'log_loss': log_loss(y_test, proba), 'n_train': len(X_train), 'n_test': len(X_test)})
    fitted[cal_name] = calibrated
    roc_data[cal_name] = roc_curve(y_test, proba)[:2]

    return pd.DataFrame(results), fitted, roc_data, (X_test, y_test)

print('Model zoo helper functions ready.')

### 4a. Run the xG model zoo (pre-shot features only, penalties excluded)

In [ ]:
xg_feature_cols = ['distance', 'angle_deg', 'is_header', 'is_left_foot', 'is_right_foot',
                    'is_open_play', 'is_set_piece', 'under_pressure']

xg_data = df[df['is_penalty'] == 0].dropna(subset=xg_feature_cols + ['goal']).copy()
X_xg = xg_data[xg_feature_cols]
y_xg = xg_data['goal']

print(f"Training xG model zoo on {len(xg_data)} non-penalty shots ({y_xg.sum()} goals)")
xg_results, xg_fitted, xg_roc_data, (xg_X_test, xg_y_test) = run_model_zoo(X_xg, y_xg, 'xG')
xg_results.sort_values('roc_auc', ascending=False).reset_index(drop=True)

### 4b. Run the PSxG model zoo (adds placement features) — only if the dataset has them

In [ ]:
if HAS_PLACEMENT:
    psxg_feature_cols = xg_feature_cols + ['dist_to_nearest_post', 'placement_height', 'on_target']
    psxg_data = df[df['is_penalty'] == 0].dropna(subset=psxg_feature_cols + ['goal']).copy()
    X_psxg = psxg_data[psxg_feature_cols]
    y_psxg = psxg_data['goal']

    print(f"Training PSxG model zoo on {len(psxg_data)} non-penalty shots ({y_psxg.sum()} goals)")
    psxg_results, psxg_fitted, psxg_roc_data, (psxg_X_test, psxg_y_test) = run_model_zoo(X_psxg, y_psxg, 'PSxG')
    display(psxg_results.sort_values('roc_auc', ascending=False).reset_index(drop=True))
else:
    psxg_results = pd.DataFrame()
    psxg_fitted, psxg_roc_data = {}, {}
    print('Skipped — no placement columns in this dataset (see note above).')

## 5. Part D — Save all results, models, and images

In [ ]:
all_results = pd.concat([xg_results, psxg_results], ignore_index=True) if len(psxg_results) else xg_results.copy()
all_results = all_results.sort_values(['task', 'roc_auc'], ascending=[True, False]).reset_index(drop=True)
all_results.to_csv(f'{RESULTS_DIR}/model_comparison_results.csv', index=False)
print(f"Saved model_comparison_results.csv ({len(all_results)} rows)")
all_results

In [ ]:
# Test-set predictions from every xG model, for inspection / auditing
pred_table = xg_X_test.copy()
pred_table['actual_goal'] = xg_y_test.values
for name, model in xg_fitted.items():
    pred_table[f'pred_{name}'] = model.predict_proba(xg_X_test)[:, 1]
pred_table.to_csv(f'{RESULTS_DIR}/xg_test_predictions.csv', index=False)
print(f"Saved xg_test_predictions.csv ({len(pred_table)} rows, {pred_table.shape[1]} columns)")
pred_table.head()

In [ ]:
# Save every fitted xG model + the best PSxG model (if it ran) with joblib
best_xg_name = xg_results.loc[xg_results['roc_auc'].idxmax(), 'model']
joblib.dump(xg_fitted[best_xg_name], f'{RESULTS_DIR}/best_xg_model__{best_xg_name.replace(" ", "_")}.joblib')
print(f"Saved best xG model: {best_xg_name}")

if len(psxg_results):
    best_psxg_name = psxg_results.loc[psxg_results['roc_auc'].idxmax(), 'model']
    joblib.dump(psxg_fitted[best_psxg_name], f'{RESULTS_DIR}/best_psxg_model__{best_psxg_name.replace(" ", "_")}.joblib')
    print(f"Saved best PSxG model: {best_psxg_name}")

### Images — bar chart comparing every model

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, better in zip(axes, ['roc_auc', 'brier', 'log_loss'], ['higher = better', 'lower = better', 'lower = better']):
    plot_df = all_results.sort_values(metric, ascending=(metric != 'roc_auc'))
    colors = sns.color_palette('viridis', len(plot_df))
    ax.barh(plot_df['model'] + ' (' + plot_df['task'] + ')', plot_df[metric], color=colors)
    ax.set_title(f'{metric} — {better}')
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/model_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### Images — ROC curves for the xG task

In [ ]:
plt.figure(figsize=(7, 7))
for name, (fpr, tpr) in xg_roc_data.items():
    auc = xg_results.loc[xg_results['model'] == name, 'roc_auc'].values[0]
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — xG models')
plt.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/roc_curves_xg.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if len(psxg_results):
    plt.figure(figsize=(7, 7))
    for name, (fpr, tpr) in psxg_roc_data.items():
        auc = psxg_results.loc[psxg_results['model'] == name, 'roc_auc'].values[0]
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves — PSxG models')
    plt.legend(fontsize=8, loc='lower right')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/roc_curves_psxg.png', dpi=150, bbox_inches='tight')
    plt.show()

### Images — feature importance from the best tree-based model

In [ ]:
tree_models = {n: m for n, m in xg_fitted.items() if hasattr(m, 'feature_importances_')}
if tree_models:
    best_tree_name = max(tree_models, key=lambda n: xg_results.loc[xg_results['model'] == n, 'roc_auc'].values[0])
    importances = tree_models[best_tree_name].feature_importances_
    imp_df = pd.DataFrame({'feature': xg_feature_cols, 'importance': importances}).sort_values('importance')

    plt.figure(figsize=(7, 5))
    plt.barh(imp_df['feature'], imp_df['importance'], color='teal')
    plt.title(f'Feature importance — {best_tree_name} (xG task)')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/feature_importance_xg.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No tree-based model with feature_importances_ available.')

### Images — confusion matrix for the best xG model

In [ ]:
best_model = xg_fitted[best_xg_name]
y_pred_labels = (best_model.predict_proba(xg_X_test)[:, 1] >= 0.5).astype(int)
cm = confusion_matrix(xg_y_test, y_pred_labels)

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(cm, display_labels=['No Goal', 'Goal']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_xg_name} (xG, threshold=0.5)')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrix_xg.png', dpi=150, bbox_inches='tight')
plt.show()

### Images — shot map and StatsBomb-xG comparison (if the columns are present)

In [ ]:
if {'x', 'y'}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot([60, 60, 120, 120, 60], [0, 80, 80, 0, 0], color='black', linewidth=1)
    ax.plot([102, 102, 120, 120, 102], [18, 62, 62, 18, 18], color='black', linewidth=1)
    ax.plot([114, 114, 120, 120, 114], [30, 50, 50, 30, 30], color='black', linewidth=1)
    ax.set_xlim(58, 122); ax.set_ylim(-2, 82); ax.axis('off')

    goals = df[df['goal'] == 1]
    misses = df[df['goal'] != 1]
    ax.scatter(misses['x'], misses['y'], s=15, color='grey', alpha=0.35, label='No goal')
    ax.scatter(goals['x'], goals['y'], s=25, color='crimson', alpha=0.8, label='Goal', edgecolors='black', linewidths=0.3)
    ax.legend(loc='lower left')
    ax.set_title(f'Shot map — {len(df)} shots')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/shot_map.png', dpi=150, bbox_inches='tight')
    plt.show()

if 'statsbomb_xg' in df.columns:
    plt.figure(figsize=(6, 6))
    best_xg_proba_full = xg_fitted[best_xg_name].predict_proba(df.loc[xg_data.index, xg_feature_cols])[:, 1]
    plt.scatter(xg_data['statsbomb_xg'] if 'statsbomb_xg' in xg_data.columns else np.nan,
                best_xg_proba_full, alpha=0.3, s=10)
    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlabel('StatsBomb xG'); plt.ylabel(f'{best_xg_name} (ours)')
    plt.title('Our best xG model vs StatsBomb xG')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/xg_vs_statsbomb.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Part E — Bonus: shot clustering + where computer vision actually fits

**Clustering (unsupervised)** — groups shots into archetypes (e.g. close-range
headers, edge-of-box strikes, one-on-ones) using only location + distance +
angle. This isn't predicting goals, it's exploring shot *types*.

In [ ]:
from sklearn.cluster import KMeans

cluster_features = ['x', 'y', 'distance', 'angle_deg']
cluster_data = df.dropna(subset=cluster_features).copy()
scaler = StandardScaler()
Xc = scaler.fit_transform(cluster_data[cluster_features])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_data['shot_cluster'] = kmeans.fit_predict(Xc)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot([60, 60, 120, 120, 60], [0, 80, 80, 0, 0], color='black', linewidth=1)
scatter = ax.scatter(cluster_data['x'], cluster_data['y'], c=cluster_data['shot_cluster'],
                      cmap='tab10', s=15, alpha=0.6)
ax.set_xlim(58, 122); ax.set_ylim(-2, 82); ax.axis('off')
ax.set_title('Shot archetypes (K-Means, 5 clusters)')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/shot_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

cluster_data.groupby('shot_cluster')[['distance', 'angle_deg', 'goal']].mean()

### Where computer vision actually applies (and why it doesn't apply to this CSV directly)

This dataset is **coordinates and categories** (shot x/y, body part, outcome)
— not pixels. CNNs and other CV models operate on image grids, so they don't
plug into `xg_psxg_shots.csv` as-is. In real xG research, CV shows up in two
different places:

1. **Freeze-frame CNNs** — some StatsBomb competitions include a
   `shot_freeze_frame` field: the (x, y) position of every player on the pitch
   at the moment of the shot. Researchers rasterize that into a 2D grid
   (like a tiny image of player density) and train a CNN on it *instead of*
   hand-built features like `dist_to_nearest_post`. This notebook doesn't do
   that, but the raw `shot_freeze_frame` column is sitting in `raw_shots.csv`
   from Part A if you download fresh — that's the door to walk through if you
   want to try it.
2. **Real video-based tracking** — pose estimation (OpenPose), object
   detection (YOLO), and multi-object tracking (DeepSORT) are what clubs and
   broadcasters use on actual match-camera footage to get player positions in
   the first place. That needs raw video, which StatsBomb's open data doesn't
   provide — this is a separate, heavier pipeline than anything here.

If you want to go down the freeze-frame route, say so and I'll add a
dedicated section for it — it's a legitimate next step, just a meaningfully
bigger piece of work than the model zoo above.

## 7. Zip everything and download

In [ ]:
zip_path = shutil.make_archive('xg_project_results', 'zip', RESULTS_DIR)
print(f"Zipped results to {zip_path}")
print('Contents:')
for f in sorted(os.listdir(RESULTS_DIR)):
    print(' -', f)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f"Not running in Colab — find your results at {zip_path} and inside {RESULTS_DIR}/")